<table><tr>
<td><b>Sapienza University of Rome</b><br>PhD Soft Skills 2026</td>
</tr></table>

# Notebook 2 — Images, and how to fool them
**5 minutes — watch this one.** Your instructor runs it on screen. You do not need to type anything, but you can run it yourself in under a minute.

> If you do want to run it: click `Copy to Drive` in the toolbar first, then run the cells in order.


Everything so far has been text. Images are the other classic case, and they fail in a way that is worth seeing with your own eyes.

We use the handwritten-digit dataset that ships inside scikit-learn — 1,797 tiny images of numbers, each 8×8 pixels. No download, no GPU, no waiting.

### 1 · Build an image classifier

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Load the built-in handwritten digits dataset from scikit-learn.
Show me 10 example images with their labels.
Then split 80/20 and train a LOGISTIC REGRESSION classifier on the
training part, and report its accuracy on the held-out 20%.
Keep the fitted model in a variable called `clf`.
Finally show a grid of the test images it got wrong, each labelled
with the true digit and what it guessed.
```

> The model type is pinned deliberately. Different classifiers behave very differently in the second half of this notebook — a random forest, for instance, *does* lose confidence under noise, which would demonstrate the opposite of the point being made.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
import matplotlib.pyplot as plt, numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

digits = load_digits()
X, y = digits.data, digits.target

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for ax, img, lab in zip(axes, digits.images, y):
    ax.imshow(img, cmap='gray_r'); ax.set_title(int(lab)); ax.axis('off')
plt.suptitle('what the machine is shown'); plt.show()

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
clf = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(f'Test accuracy: {accuracy_score(yte, pred):.1%}')

bad = np.where(pred != yte)[0][:10]
fig, axes = plt.subplots(1, len(bad), figsize=(12, 1.8))
for ax, i in zip(np.atleast_1d(axes), bad):
    ax.imshow(Xte[i].reshape(8, 8), cmap='gray_r')
    ax.set_title(f'{yte[i]}\u2192{pred[i]}', fontsize=9); ax.axis('off')
plt.suptitle('true \u2192 guessed'); plt.show()

---
## Now break it

The accuracy above is around 97%. Here is the part that matters.

We add random noise to the test images — the kind of degradation any real scan has — and then look at **two** things: how far accuracy falls, and how far *confidence* falls.

The comparison that matters is confidence **before** versus **after**. Watch those two numbers, not the others.

### 2 · Add noise, and compare confidence before and after

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Add Gaussian noise with standard deviation 4.0 to the test images
(the pixel scale runs 0 to 16), and clip the result back into that range.

Then print a small table comparing CLEAN versus NOISY:
  - accuracy on each
  - the model's MEAN CONFIDENCE on each (the highest predicted
    probability, averaged over all test images)
Print how many percentage points each one dropped.

Then plot the distribution of the model's confidence in its WRONG
answers on the noisy images, and show a few noisy images next to
their clean originals.
```

> The noise level is pinned because the demo only works in a narrow band: at sd=2 accuracy barely moves and there is nothing to see; at sd=8 the digits are destroyed and the failure is unsurprising.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
rng = np.random.default_rng(0)
Xte_noisy = np.clip(Xte + rng.normal(0, 4.0, Xte.shape), 0, 16)

pred_c, pred_n = clf.predict(Xte), clf.predict(Xte_noisy)
conf_c = clf.predict_proba(Xte).max(axis=1)
conf_n = clf.predict_proba(Xte_noisy).max(axis=1)

acc_c, acc_n = accuracy_score(yte, pred_c), accuracy_score(yte, pred_n)

print(f'{"":18s}{"clean":>10s}{"noisy":>10s}{"change":>12s}')
print(f'{"accuracy":18s}{acc_c:>10.1%}{acc_n:>10.1%}{acc_n - acc_c:>+11.1f} pts')
print(f'{"mean confidence":18s}{conf_c.mean():>10.1%}{conf_n.mean():>10.1%}{100 * (conf_n.mean() - conf_c.mean()):>+11.1f} pts')
print()
print('Accuracy collapsed. Confidence barely moved.')
print('The model has no way of telling you it has left familiar ground.')
print()
wrong = pred_n != yte
print(f'Of its WRONG answers on noisy images, {(conf_n[wrong] > 0.9).mean():.0%} were held with over 90% confidence.')

plt.figure(figsize=(7, 3.5))
plt.hist(conf_n[wrong], bins=20, color='#B3283C')
plt.xlabel('model confidence in its WRONG answer'); plt.ylabel('count')
plt.title('It is confidently wrong'); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for j in range(8):
    axes[0, j].imshow(Xte[j].reshape(8, 8), cmap='gray_r'); axes[0, j].axis('off')
    axes[1, j].imshow(Xte_noisy[j].reshape(8, 8), cmap='gray_r'); axes[1, j].axis('off')
axes[0, 0].set_title('clean', fontsize=9, loc='left')
axes[1, 0].set_title('noisy', fontsize=9, loc='left')
plt.suptitle('the same test images, before and after noise')
plt.show()

---
## The lesson

Accuracy falls off a cliff. Confidence strolls down a gentle slope. The model does not fail loudly — it fails **quietly and confidently**.

Now think about your own material: manuscripts photographed in different lighting, documents digitised by different institutions on different scanners, recordings made in different rooms. Every one of those is the noise in this experiment.

> **Playbook line:** when a model meets material unlike its training data, accuracy drops and confidence does not. Confidence is never evidence.